# 📊 evaluation.py – Pipeline de Avaliação RAG

Este módulo implementa um pipeline completo de avaliação para sistemas **RAG (Retrieval-Augmented Generation)**, cobrindo desde o *chunking* até métricas de geração e julgamento por LLM.

Fluxo: Documentos → Chunking → Embeddings → ChromaDB → Retrieval → Reranking (MMR) → Prompt → LLM → Avaliação.

## Etapas

- **Chunking:** diferentes estratégias (chunk_size, overlap, semântico, por sentença).
- **Armazenamento:** geração de embeddings e indexação no ChromaDB.
- **Retrieval:** busca por similaridade (cosine similarity) entre query e chunks.
- **Reranking:** MMR (Maximal Marginal Relevance).
- **Métricas de Retrieval:** Recall@k, Precision@k, F1@k, NDCG@k.
- **Prompt Engineering:** variação e controle de instruções.
- **LLM as Judge:** avaliação automática de relevância e coerência.
- **Métricas de Geração:** BLEU e ROUGE.

**Objetivo: analisar impacto do chunking, retrieval e geração no desempenho do RAG.**


## Importação das bibliotecas : 

In [1]:
from PyPDF2 import PdfReader
import warnings
warnings.filterwarnings("ignore")
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import numpy as np
import time
import math
import chromadb
from chromadb.config import Settings
from sklearn.metrics.pairwise import cosine_similarity
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
import json
import re

## Leitura :

In [2]:
pdf_path = "Pensamento_maquina.pdf"

reader = PdfReader(pdf_path)

print(f"Numéro de páginas : {len(reader.pages)}")

full_text = ""

for i, page in enumerate(reader.pages):
    text = page.extract_text()
    full_text += text + "\n"

print("Tamanho total do texto:", len(full_text))


Numéro de páginas : 4
Tamanho total do texto: 9140


## Chunking :

### Método 1 :

In [3]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,     
    chunk_overlap=150,  
    separators=["\n\n", "\n", ".", " ", ""]
)

chunks = text_splitter.split_text(full_text)

print("Número total de chunks:", len(chunks))

for i, chunk in enumerate(chunks[:2]):
    print("\n====================")
    print(f"CHUNK {i}")
    print("====================")
    print(chunk)


Número total de chunks: 25

CHUNK 0
O Pensamento Humano e o Pensamento das  
Máquinas
Introdução
Desde os primórdios da ﬁlosoﬁa, o ser humano busca compreender a própria mente.  
Perguntas como o que é pensar? , como surge o conhecimento?  e o que nos torna conscientes?  
atravessam séculos de reﬂexão ﬁlosóﬁca, cientíﬁca e cultural. Com o avanço da  
tecnologia e, especialmente, com o surgimento da Inteligência Artiﬁcial, essas questões  
ganharam uma nova dimensão: ao criar máquinas capazes de executar tarefas cognitivas,

CHUNK 1
ganharam uma nova dimensão: ao criar máquinas capazes de executar tarefas cognitivas,  
o ser humano passou a se perguntar se estaria, de alguma forma, reproduzindo o próprio  
pensamento.
A relação entre o pensamento humano e o pensamento das máquinas não é apenas  
técnica. Trata-se de uma questão conceitual, ﬁlosóﬁca e ética. Comparar esses dois tipos  
de “pensamento” nos obriga a reﬂetir sobre os limites da tecnologia e, ao mesmo tempo,  
sobre a nature

- O RecursiveCharacterTextSplitter é usado quando queremos chunks coerentes semanticamente, mas ainda com controle rígido de tamanho.
Ele tenta dividir o texto de forma hierárquica:

- Primeiro por \n\n (parágrafos), depois por \n (linhas) , depois por ".", depois por espaço e por último, caractere por caractere

**Ou seja: ele prioriza manter a estrutura natural do texto antes de cortar brutalmente pelo tamanho.**

## Modelo de embedding : 

In [4]:
st = SentenceTransformer("all-MiniLM-L6-v2")

print("Gerando embeddings dos chunks...\n")

chunk_embeddings = st.encode(chunks)

print("Número de embeddings:", len(chunk_embeddings))
print("Shape de UM embedding:", chunk_embeddings[0].shape)


Gerando embeddings dos chunks...

Número de embeddings: 25
Shape de UM embedding: (384,)


Usamos para fazer os **embeddings** (Transformar chunks em **vetores** com **significado semântico**) o modelo **all-MiniLM-L6-v2**.
Os 25 vetores possuem 384 dimensões


### Dense retrieval

In [6]:
queries = [
    "Explique como a IA usa matemática para processar informação",
    "Como redes neurais artificiais se inspiram no cérebro humano?",
    "Quais tarefas as máquinas realizam melhor que os humanos?"
]

top_k = 1

for idx, query in enumerate(queries, start=1):

    query_embedding = st.encode([query])
    similarities = cosine_similarity(query_embedding, chunk_embeddings)[0]

    top_indices = np.argsort(similarities)[::-1][:top_k]

    print(f"\n==============================")
    print(f"Query {idx}: {query}")

    for rank, chunk_idx in enumerate(top_indices, start=1):
        print(f"\nTop {rank}")
        print(f"Chunk index: {chunk_idx}")
        print(f"Similaridade: {similarities[chunk_idx]:.4f}")
        print(f"{chunks[chunk_idx][:500]}...\n")



Query 1: Explique como a IA usa matemática para processar informação

Top 1
Chunk index: 8
Similaridade: 0.5838
Artiﬁcial são projetados para processar grandes volumes de dados, identiﬁcar padrões e  
produzir respostas ou ações com base nesses padrões.
Diferentemente do ser humano, a máquina não possui consciência, emoções ou intenções  
próprias. Quando uma IA “decide” algo, essa decisão é o resultado de cálculos  
matemáticos realizados a partir de parâmetros deﬁnidos durante o treinamento. Não há  
compreensão subjetiva do que está sendo feito, apenas a execução de regras estatísticas....


Query 2: Como redes neurais artificiais se inspiram no cérebro humano?

Top 1
Chunk index: 11
Similaridade: 0.6884
não entende signiﬁcados; ela manipula representações numéricas associadas a padrões  
observados.
A inspiração no cérebro humano
Grande parte do desenvolvimento da Inteligência Artiﬁcial foi inspirada no  
funcionamento do cérebro humano. Redes neurais artiﬁciais, por exemplo, rece

Estes são as chunks que mais posssuem similaridade com as queries (Perguntas), a métrica é feita através da **similaridade do cosseno**, que usa da propriedade do ângulo entre vetores da álgebra linear para avaliar o quão próximo dois vetores estão no espaço vetorial.

A  variável **top_k** permite pesquisarmos k vetores e verificar o ranking das similaridades.

## Armazenando no banco de dados vetorial (ChromaDB) :

In [5]:
client = chromadb.Client()

collection = client.create_collection(name="rag_test")

print("Collection criada.")

ids = [f"chunk_{i}" for i in range(len(chunks))]

collection.add(
    documents=chunks,
    embeddings=chunk_embeddings.tolist(),
    ids=ids
)

print("Chunks inseridos no ChromaDB.")
print("Total armazenado:", len(ids))



Collection criada.
Chunks inseridos no ChromaDB.
Total armazenado: 25


## carregando o modelo da llm :

In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "Qwen/Qwen2.5-0.5B-Instruct"

tokenizer_llm = AutoTokenizer.from_pretrained(model_name)

model_llm = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if device == "cuda" else torch.float32
).to(device)


In [8]:
def precision_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & set(relevant)) / k

def recall_at_k(retrieved, relevant, k):
    return len(set(retrieved[:k]) & set(relevant)) / len(relevant)

def f1(p, r):
    if p + r == 0:
        return 0
    return 2 * (p * r) / (p + r)

def mrr(retrieved, relevant):
    for rank, doc_id in enumerate(retrieved, start=1):
        if doc_id in relevant:
            return 1 / rank
    return 0

def ndcg_at_k(retrieved, relevant, k):
    dcg = 0
    for i in range(len(retrieved[:k])):
        if retrieved[i] in relevant:
            dcg += 1 / math.log2(i + 2)
    ideal_hits = min(len(relevant), k)
    idcg = sum(1 / math.log2(i + 2) for i in range(ideal_hits))
    return dcg / idcg if idcg > 0 else 0


queries = [
    "Explique como a IA usa matemática para processar informação",
    "Como redes neurais artificiais se inspiram no cérebro humano?",
    "Quais tarefas as máquinas realizam melhor que os humanos?"
]

k = 3

ground_truth = {
    0: ["chunk_8", "chunk_9", "chunk_10"],
    1: ["chunk_11", "chunk_12"],
    2: ["chunk_19", "chunk_15"]
}


all_precisions, all_recalls, all_f1s, all_mrrs, all_ndcgs, latencies = [], [], [], [], [], []

for q_idx, query in enumerate(queries):

    start_time = time.time()

    query_embedding = st.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k
    )

    retrieved_ids = results["ids"][0]

    latency = time.time() - start_time
    latencies.append(latency)

    relevant_ids = ground_truth[q_idx]

    p = precision_at_k(retrieved_ids, relevant_ids, k)
    r = recall_at_k(retrieved_ids, relevant_ids, k)
    f = f1(p, r)
    m = mrr(retrieved_ids, relevant_ids)
    n = ndcg_at_k(retrieved_ids, relevant_ids, k)

    all_precisions.append(p)
    all_recalls.append(r)
    all_f1s.append(f)
    all_mrrs.append(m)
    all_ndcgs.append(n)

    print("\n==============================")
    print("Query:", query)
    print("Retrieved IDs:", retrieved_ids)
    print("Chunks:")
    for doc in results["documents"][0]:
        print("-", doc[:200], "...")
        print("\n")

    print(f"Precision@{k}: {p:.4f}")
    print(f"Recall@{k}: {r:.4f}")
    print(f"F1@{k}: {f:.4f}")
    print(f"MRR: {m:.4f}")
    print(f"nDCG@{k}: {n:.4f}")
    print(f"Tempo de resposta: {latency:.6f} segundos")


print("\n==============================")
print("RESULTADOS FINAIS (CHUNKS)")
print("==============================")
print(f"Mean Precision@{k}: {np.mean(all_precisions):.4f}")
print(f"Mean Recall@{k}: {np.mean(all_recalls):.4f}")
print(f"Mean F1@{k}: {np.mean(all_f1s):.4f}")
print(f"Mean MRR: {np.mean(all_mrrs):.4f}")
print(f"Mean nDCG@{k}: {np.mean(all_ndcgs):.4f}")
print(f"Latência média: {np.mean(latencies):.6f} segundos")



Query: Explique como a IA usa matemática para processar informação
Retrieved IDs: ['chunk_8', 'chunk_24', 'chunk_3']
Chunks:
- Artiﬁcial são projetados para processar grandes volumes de dados, identiﬁcar padrões e  
produzir respostas ou ações com base nesses padrões.
Diferentemente do ser humano, a máquina não possui consciê ...


- como uma colaboração — uma parceria entre o pensamento consciente e o cálculo  
automatizado. ...


- A natureza do pensamento humano
O pensamento humano é um fenômeno extremamente complexo. Ele não pode ser  
reduzido a uma sequência linear de operações lógicas, pois envolve múltiplas dimensões  
que ...


Precision@3: 0.3333
Recall@3: 0.3333
F1@3: 0.3333
MRR: 1.0000
nDCG@3: 0.4693
Tempo de resposta: 12.558811 segundos

Query: Como redes neurais artificiais se inspiram no cérebro humano?
Retrieved IDs: ['chunk_11', 'chunk_12', 'chunk_2']
Chunks:
- não entende signiﬁcados; ela manipula representações numéricas associadas a padrões  
observados.
A inspiraç


Este script avalia a qualidade do **retrieval** de um sistema RAG utilizando métricas clássicas.
Foi verificado manualmente quais chunks respondiam melhor às queries em ordem de qualidade e postas em ground truth.

**Ground Truth :**
Dicionário que define quais chunks são considerados as melhores respostas com relação às queries.

**precision_at_k:**
Mede quantos documentos relevantes aparecem entre os top-k recuperados.  

**recall_at_k :**
Mede quantos documentos relevantes foram recuperados dentre todos os relevantes existentes.

**f1 :**
Média harmônica entre precision e recall.

**mrr (Mean Reciprocal Rank) :**
Avalia a posição do primeiro documento relevante na lista.  
Quanto mais cedo aparecer, maior o score.

**ndcg_at_k :** 
Avalia a qualidade do ranking considerando a posição dos relevantes.  
Recompensa documentos relevantes que aparecem mais no topo. 

**Latência :**
Tempo de resposta para cada query.

RESULTADOS FINAIS (CHUNKS)
==============================
Mean Precision@3: 0.4444
Mean Recall@3: 0.6111
Mean F1@3: 0.5111
Mean MRR: 1.0000
Mean nDCG@3: 0.6941
Latência média: 1.387487 segundos


### Prompt engineering : 

In [15]:
def generate_answer(prompt):

    messages = [
        {"role": "system", "content": "Responda sempre em português e seja objetivo."},
        {"role": "user", "content": prompt}
    ]

    text = tokenizer_llm.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer_llm(
        text,
        return_tensors="pt"
    )

    outputs = model_llm.generate(
        **inputs,
        max_new_tokens=200,
        temperature=0.7,
        top_p=0.9,
        do_sample=True
    )

    response = tokenizer_llm.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    )

    return response.strip()


## Definindo o RAG :

In [16]:
def mini_rag(query, n_results=3):

    query_embedding = st.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results
    )

    retrieved_chunks = results["documents"][0]

    context = "\n".join(retrieved_chunks[:2])

    prompt = f"""
Use apenas o contexto para responder.

Contexto:
{context}

Pergunta:
{query}
"""

    answer = generate_answer(prompt)

    return {
        "query": query,
        "context": context,
        "answer": answer
    }


### Teste :

In [13]:
generate_answer("Me diga 3 principais diferenças entre o cérebro humano e o processamento das máquinas, seja breve")

'Aqui estão três principais diferenças entre o cérebro humano e o processo de processamento por máquinas:\n\n1. **Cortizão Neurofísico**: O cérebro humano possui um cortizão neurofísico que é mais complexo e multifuncional. Ele é responsável pela formação cognitiva, memória, emoção, pensamento, interações sociais e outros processos cognitivos essenciais.\n\n2. **Lógica Computacional**: Os sistemas de computador são baseados na lógica computacional, onde as informações são manipuladas através de operações matemáticas e algoritmos. Por outro lado, o cérebro humano tem uma lógica computacional muito mais complexa, incorporando conceitos como a consciência, a inteligência emocional e os padrões cognitivos.\n\n3. **Repetição e Desenvolvimento**: O cérebro humano é cap'

## LLM-AS-JUDGE : 

In [ ]:
def extrair_json(texto):
    match = re.search(r"\{.*\}", texto, re.DOTALL)
    if match:
        try:
            return json.loads(match.group())
        except:
            return None
    return None


def avaliar_relevancia(pergunta, resposta):

    prompt = f"""
Avalie de 0 a 10 o quanto a resposta responde corretamente a pergunta.

Pergunta:
{pergunta}

Resposta:
{resposta}

Retorne apenas JSON:
{{"relevancia": numero}}
"""

    result = generate_answer(prompt)
    return extrair_json(result)



def avaliar_coerencia(contexto, resposta):
    prompt = f"""
Avalie de 0 a 10 se a resposta faz sentido internamente e mantém consistência lógica.
Contexto:
{contexto}

Resposta:
{resposta}

Retorne apenas JSON:
{{"coerencia": numero}}
"""
    result = generate_answer(prompt)
    return extrair_json(result)



def avaliar_correcao(pergunta, resposta):

    prompt = f"""
Avalie de 0 a 10 se a resposta está factualmente correta
com base em conhecimento geral.

Pergunta:
{pergunta}

Resposta:
{resposta}

Retorne apenas JSON:
{{"correcao": numero}}
"""

    result = generate_answer(prompt)
    return extrair_json(result)



def avaliar_completude(pergunta, resposta):

    prompt = f"""
Avalie de 0 a 10 se a resposta está completa
ou se faltam partes importantes.

Pergunta:
{pergunta}

Resposta:
{resposta}

Retorne apenas JSON:
{{"completude": numero}}
"""

    result = generate_answer(prompt)
    return extrair_json(result)


def avaliar_resposta_rag(pergunta, contexto, resposta):

    return {
        "relevancia": avaliar_relevancia(pergunta, resposta),
        "coerencia": avaliar_coerencia(contexto, resposta),
        "correcao": avaliar_correcao(pergunta, resposta),
        "completude": avaliar_completude(pergunta, resposta)
    }


"LLM as Judge" é uma abordagem em que uma **Large Language Model** atua como avaliadora automática das respostas geradas pelo sistema.

- **Relevância:** a resposta está relacionada à pergunta?
-**Coerência:** a resposta faz sentido internamente?
- **Correção:** não contém erros de linguagem ou lógica?
- **Completude:** cobre todos os aspectos importantes da pergunta?

Outras métricas poderiam ser usadas como :
- **Fidelidade / Factually Correct:** as informações estão corretas e precisas?
- **Clareza:** a resposta é clara e fácil de entender?
- **Concisão / Brevidade:** responde de forma direta, sem enrolação?

**Bastaria criar o prompt**

### Avaliando : 

In [15]:
resultado = mini_rag("Explique redes neurais")

avaliacao = avaliar_resposta_rag(
    pergunta=resultado["query"],
    contexto=resultado["context"],
    resposta=resultado["answer"]
)

print(avaliacao)


{'relevancia': {'relevancia': 8}, 'fidelidade': {'fidelidade': 8}, 'correcao': {'correcao': 9}, 'completude': {'completude': 8.5}}


## BLUE/ROUGE : 

In [6]:
import torch
import numpy as np
from transformers import AutoModelForCausalLM, AutoTokenizer
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer


model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

device = "cuda" if torch.cuda.is_available() else "cpu"

tokenizer_llm = AutoTokenizer.from_pretrained(model_id)
model_llm = AutoModelForCausalLM.from_pretrained(model_id).to(device)
model_llm.eval()


ground_truth_answers = {
    "diferencie os métodos de como humanos e máquinas processam informação":
        "O ser humano pensa com consciência, emoção e significado, enquanto a máquina processa informações de forma matemática e estatística.",
    "A maquina tem metacognição?":
        "A máquina não possui metacognição genuína; qualquer forma de autoavaliação é previamente programada ou estatisticamente induzida."
}

def generate_answer(query, k=3):

    query_embedding = st.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    results = collection.query(
        query_embeddings=query_embedding.tolist(),
        n_results=k
    )

    retrieved_chunks = results["documents"][0]
    context = " ".join(retrieved_chunks)

    prompt = f"""
Responda à pergunta usando apenas o contexto.
Em UMA única frase direta e curta.
Apenas a resposta. 
A resposta precisa estar entre 100 e 150 caracteres.
Dê resposta completa.

Contexto:
{context}

Pergunta:
{query}

Resposta:
"""

    inputs = tokenizer_llm(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(device)

    with torch.no_grad():
        outputs = model_llm.generate(
            **inputs,
            max_new_tokens=60,
            temperature=0.0,
            do_sample=False
        )

    generated_text = tokenizer_llm.decode(outputs[0], skip_special_tokens=True)

    answer = generated_text[len(prompt):].strip()
    return answer


smooth = SmoothingFunction().method1
rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

bleu_scores = []
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

print("\n==============================")
print("AVALIAÇÃO RAG - TinyLlama")
print("==============================")

for query, reference_answer in ground_truth_answers.items():

    generated_answer = generate_answer(query, k=3)

    reference_tokens = [reference_answer.split()]
    generated_tokens = generated_answer.split()

    bleu = sentence_bleu(
        reference_tokens,
        generated_tokens,
        smoothing_function=smooth
    )
    bleu_scores.append(bleu)

    scores = rouge.score(reference_answer, generated_answer)

    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

    print("\n------------------------------")
    print("Pergunta:", query)
    print("Resposta gerada:", generated_answer)
    print("Resposta correta:", reference_answer)
    print("BLEU:", round(bleu, 4))
    print("ROUGE-1:", round(scores["rouge1"].fmeasure, 4))
    print("ROUGE-2:", round(scores["rouge2"].fmeasure, 4))
    print("ROUGE-L:", round(scores["rougeL"].fmeasure, 4))

print("\n==============================")
print("MÉDIAS FINAIS")
print("==============================")

print("BLEU médio:", round(np.mean(bleu_scores), 4))
print("ROUGE-1 médio:", round(np.mean(rouge1_scores), 4))
print("ROUGE-2 médio:", round(np.mean(rouge2_scores), 4))
print("ROUGE-L médio:", round(np.mean(rougeL_scores), 4))



AVALIAÇÃO RAG - TinyLlama

------------------------------
Pergunta: diferencie os métodos de como humanos e máquinas processam informação
Resposta gerada: O pensamento humano e o pensamento das máquinas são diferentes em muitos aspectos, mas  
têm em comum a capacidade de processar informação. O pensamento humano é um processo  
de compreensão e reflexão, enquanto
Resposta correta: O ser humano pensa com consciência, emoção e significado, enquanto a máquina processa informações de forma matemática e estatística.
BLEU: 0.0092
ROUGE-1: 0.3492
ROUGE-2: 0.0656
ROUGE-L: 0.254

------------------------------
Pergunta: A maquina tem metacognição?
Resposta gerada: Não. A máquina não possui essa metacognição genuína; qualquer forma de “autoavaliação” é
previamente programada ou estatisticamente induzida.

Pergunta:
Por que a inteligência artificial
Resposta correta: A máquina não possui metacognição genuína; qualquer forma de autoavaliação é previamente programada ou estatisticamente induzida.

Este script avalia respostas geradas por um **pipeline RAG** (Qwen + ChromaDB) comparando-as com respostas de referência (ground truth) que peguei diretamente do PDF em questão (então é a resposta factual).
Pedi ao prompt para ele ser direto e ter entre 100 e 150 caracteres para ficar mais próximo á verdade factual e o
BLEU e o ROUGE ser mais preciso (ele poderia ser assertivo mas ser prolixo ou gerar muito texto )

**Cálculo de métricas**
   - **BLEU**: mede a sobreposição de n-gramas entre a resposta gerada e a referência.
   - **ROUGE**: mede similaridade baseada em recall de n-gramas e subsequências:

**BLEU :**
- **Propósito:** avaliar similaridade entre a resposta gerada e a referência.
- **Como funciona:** mede sobreposição de n-gramas (sequências de palavras) entre gerada e referência.
- **Interpretação:** 0 = sem coincidência, 1 = perfeita coincidência.  
- **Nota:** mais usado em tradução e geração de texto.

**ROUGE:**
- **Propósito:** avaliar cobertura e similaridade de conteúdo.
- **Como funciona:**
  - ROUGE-1 → unigramas (palavras individuais)
  - ROUGE-2 → bigramas (pares de palavras)
  - ROUGE-L → subsequência mais longa em comum
- **Interpretação:** 0 a 1, onde 1 = resposta gerada contém exatamente o conteúdo da referência.


Podemos verificar que : 
Na primeira resposta o tinyllama acabou dando mais informações que o necessário e não concluiu bem o raciocínio

Na segunda resposta o modelo deu uma resposta muito acurada com a verdade.


**MÉDIAS FINAIS :**

BLEU médio: 0.2611

ROUGE-1 médio: 0.5692

ROUGE-2 médio: 0.4073

ROUGE-L médio: 0.5379

É importante ressaltar que perguntas mais vagas como a primeira tem um score menor pois são encontradas em vários chunks, já a segunda query tem uma resposta mais bem definida e portanto seu score é maior.

Quanto melhor o modelo melhor fica a avaliação desta métrica. 